In [1]:
import wrds
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import duckdb

db = wrds.Connection(wrds_username='sniperw')
data_dir = "data"
# db.list_libraries()
# db.list_tables("crsp")
# db.list_tables("optionm_all")

Loading library list...
Done


In [11]:
###################
# CRSP Block      #
###################
# added exchcd=-2,-1,0 to address the issue that stocks temp stopped trading
# without exchcd=-2,-1, 0 the non-trading months will be tossed out in the output
# Code  Definition
# -2 Halted by the NYSE or AMEX
# -1 Suspended by the NYSE, AMEX, or NASDAQ
# 0 Not Trading on NYSE, AMEX, or NASDAQ
# 1 New York Stock Exchange
# 2 American Stock Exchange
# 3 NASDAQ
# For common share (share code 10 & 11)

crsp_names = db.raw_sql("""
    select
        permno,
        permco,
        ticker,
        ncusip,
        cusip,
        comnam,
        siccd,
        shrcd,
        exchcd,
        hexcd,
        shrcls,
        namedt,
        nameenddt,
        st_date,
        end_date
    from crsp.stocknames
    where shrcd in (10, 11)
      and exchcd in (1, 2, 3)
      and nameenddt >= '2000-01-01'
      and namedt <= '2025-12-31'
""", date_cols=["namedt", "nameenddt", "st_date", "end_date"])

print(crsp_names.head())
print(crsp_names.shape)

permno_list = crsp_names["permno"].dropna().astype(int).tolist()
print(f"# permnos = {len(permno_list)}")

crsp_names.to_csv(f"{data_dir}/crsp_common_stocknames_2000_2025.csv", index=False)

   permno  permco ticker    ncusip     cusip           comnam  siccd  shrcd  \
0   10001    7953   EWST  29274A10  36720410  ENERGY WEST INC   4920     11   
1   10001    7953   EWST  29274A20  36720410  ENERGY WEST INC   4920     11   
2   10001    7953   EGAS  29269V10  36720410       ENERGY INC   4920     11   
3   10001    7953   EGAS  29269V10  36720410       ENERGY INC   4925     11   
4   10001    7953   EGAS  36720410  36720410  GAS NATURAL INC   4925     11   

   exchcd  hexcd shrcls     namedt  nameenddt    st_date   end_date  
0       3      2   <NA> 1993-11-22 2008-02-04 1986-01-09 2017-08-03  
1       3      2   <NA> 2008-02-05 2009-08-03 1986-01-09 2017-08-03  
2       3      2   <NA> 2009-08-04 2009-12-17 1986-01-09 2017-08-03  
3       2      2   <NA> 2009-12-18 2010-07-08 1986-01-09 2017-08-03  
4       2      2   <NA> 2010-07-09 2017-08-03 1986-01-09 2017-08-03  
(24383, 15)
# permnos = 24383


In [17]:
def fetch_crsp_dsf_monthly(
    db,
    permno_list,
    data_dir="data",
    start_year=2000,
    end_year=2025,
    permno_chunk_size=500,
    min_abs_prc=None,
    replace=False,
):
    out_dir = Path(f"{data_dir}/crsp_dsf_monthly")
    out_dir.mkdir(parents=True, exist_ok=True)

    permno_list = [int(x) for x in permno_list if pd.notna(x)]

    # Build month list
    months = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            months.append((year, month))

    month_pbar = tqdm(months, desc="Months", unit="month")

    for year, month in month_pbar:
        ym = f"{year}_{month:02d}"
        out_file = out_dir / f"crsp_dsf_{ym}.csv"

        if out_file.exists() and not replace:
            month_pbar.set_postfix_str(f"skip {ym}")
            continue

        start_date = pd.Timestamp(year=year, month=month, day=1)
        end_date = start_date + pd.offsets.MonthEnd(1)

        monthly_parts = []

        chunk_ranges = range(0, len(permno_list), permno_chunk_size)
        chunk_pbar = tqdm(
            chunk_ranges,
            desc=f"{year}-{month:02d}",
            unit="chunk",
            leave=False,
        )

        for i in chunk_pbar:
            sub_permnos = permno_list[i:i + permno_chunk_size]
            in_clause = ",".join(str(x) for x in sub_permnos)

            prc_filter = ""
            if min_abs_prc is not None:
                prc_filter = f" and abs(prc) >= {float(min_abs_prc)}"

            sql = f"""
                select
                    permno,
                    date,
                    prc,
                    ret,
                    retx,
                    shrout,
                    vol
                from crsp.dsf
                where date between '{start_date.date()}' and '{end_date.date()}'
                  and permno in ({in_clause})
                  {prc_filter}
            """

            df_part = db.raw_sql(sql, date_cols=["date"])
            monthly_parts.append(df_part)

        if monthly_parts:
            df_month = pd.concat(monthly_parts, ignore_index=True)
        else:
            df_month = pd.DataFrame(
                columns=["permno", "date", "prc", "ret", "retx", "shrout", "vol"]
            )

        df_month.to_csv(out_file, index=False)
        month_pbar.set_postfix_str(f"saved {ym} ({len(df_month):,} rows)")


def get_common_stock_permnos(
    db,
    start_date="2000-01-01",
    end_date="2025-12-31",
):
    """
    Get CRSP common-stock PERMNO universe from stocknames.
    """
    permnos = db.raw_sql(f"""
        select distinct permno
        from crsp.stocknames
        where shrcd in (10, 11)
          and exchcd in (1, 2, 3)
          and nameenddt >= '{start_date}'
          and namedt <= '{end_date}'
        order by permno
    """)
    return permnos["permno"].dropna().astype(int).tolist()

In [ ]:
fetch_crsp_dsf_monthly(
        db=db,
        permno_list=permno_list,
        data_dir="data",
        start_year=2024,
        end_year=2024,
        permno_chunk_size=500,
        min_abs_prc=None,
        replace=False,
    )

In [19]:
# legacy csv version
"""def combine_monthly_crsp_files(
    data_dir="data",
    output_file="crsp_daily_common.csv",
):
    out_dir = Path(f"{data_dir}/crsp_dsf_monthly")
    files = sorted(out_dir.glob("crsp_dsf_*.csv"))

    if not files:
        raise FileNotFoundError(f"No monthly files found in {out_dir}")

    dfs = []
    for f in tqdm(files, desc="Combining", unit="file"):
        dfs.append(pd.read_csv(f, parse_dates=["date"]))

    df = pd.concat(dfs, ignore_index=True)
    df.to_csv(Path(data_dir) / output_file, index=False)
    return df"""

def combine_monthly_crsp_files(
    data_dir="data",
    output_file="crsp_daily_common.parquet",
    compression="zstd",
):
    out_dir = Path(f"{data_dir}/crsp_dsf_monthly")
    if not out_dir.exists():
        raise FileNotFoundError(f"Directory does not exist: {out_dir}")

    csv_glob = out_dir / "crsp_dsf_*.csv"
    files = sorted(out_dir.glob("crsp_dsf_*.csv"))
    if not files:
        raise FileNotFoundError(f"No monthly CSV files found in {out_dir}")

    output_path = Path(data_dir) / output_file
    output_path.parent.mkdir(parents=True, exist_ok=True)

    con = duckdb.connect()

    try:
        con.execute(f"""
            COPY (
                SELECT
                    CAST(permno AS BIGINT) AS permno,
                    CAST(date AS DATE) AS date,
                    CAST(prc AS DOUBLE) AS prc,
                    CAST(ret AS DOUBLE) AS ret,
                    CAST(retx AS DOUBLE) AS retx,
                    CAST(shrout AS DOUBLE) AS shrout,
                    CAST(vol AS DOUBLE) AS vol
                FROM read_csv_auto(
                    '{csv_glob.as_posix()}',
                    union_by_name = true,
                    header = true
                )
                ORDER BY date, permno
            )
            TO '{output_path.as_posix()}'
            (FORMAT PARQUET, COMPRESSION '{compression}')
        """)
    finally:
        con.close()

    return output_path

combine_monthly_crsp_files()

WindowsPath('data/crsp_daily_common.parquet')

In [20]:
def build_crsp_id_master(
    stocknames_file="data/crsp_common_stocknames_2000_2025.csv",
    output_file="data/crsp_id_master_2000_2025.csv",
):
    df = pd.read_csv(
        stocknames_file,
        parse_dates=["namedt", "nameenddt", "st_date", "end_date"]
    )

    # Standardize identifiers
    for col in ["ticker", "ncusip", "cusip", "comnam"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()

    # Prefer NCUSIP for historical matching; fall back to CUSIP
    df["cusip_hist"] = df["ncusip"].replace({"NAN": pd.NA, "NONE": pd.NA})
    if "cusip" in df.columns:
        df["cusip_hist"] = df["cusip_hist"].fillna(df["cusip"])

    df["cusip8"] = df["cusip_hist"].astype(str).str[:8]
    df["cusip6"] = df["cusip_hist"].astype(str).str[:6]

    keep_cols = [
        "permno", "permco", "ticker", "ncusip", "cusip", "cusip_hist",
        "cusip8", "cusip6", "comnam", "siccd", "shrcd", "exchcd",
        "hexcd", "shrcls", "namedt", "nameenddt"
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    id_master = (
        df[keep_cols]
        .drop_duplicates()
        .sort_values(["permno", "namedt", "nameenddt"])
        .reset_index(drop=True)
    )

    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    id_master.to_csv(output_file, index=False)

    return id_master


id_master = build_crsp_id_master()
print(id_master.head())
print(id_master.shape)

   permno  permco ticker    ncusip     cusip cusip_hist    cusip8  cusip6  \
0   10001    7953   EWST  29274A10  36720410   29274A10  29274A10  29274A   
1   10001    7953   EWST  29274A20  36720410   29274A20  29274A20  29274A   
2   10001    7953   EGAS  29269V10  36720410   29269V10  29269V10  29269V   
3   10001    7953   EGAS  29269V10  36720410   29269V10  29269V10  29269V   
4   10001    7953   EGAS  36720410  36720410   36720410  36720410  367204   

            comnam  siccd  shrcd  exchcd  hexcd shrcls     namedt  nameenddt  
0  ENERGY WEST INC   4920     11       3      2    NaN 1993-11-22 2008-02-04  
1  ENERGY WEST INC   4920     11       3      2    NaN 2008-02-05 2009-08-03  
2       ENERGY INC   4920     11       3      2    NaN 2009-08-04 2009-12-17  
3       ENERGY INC   4925     11       2      2    NaN 2009-12-18 2010-07-08  
4  GAS NATURAL INC   4925     11       2      2    NaN 2010-07-09 2017-08-03  
(24383, 16)


In [27]:
def preview_table(db, table, n=5):
    df = db.raw_sql(f"select * from optionm_all.{table} limit {n}")
    print(f"\n=== {table} ===")
    print(df.columns.tolist())
    print(df.head())
    return df

secnmd = preview_table(db, "secnmd")
optionmnames = preview_table(db, "optionmnames")
secprd = preview_table(db, "secprd")
opprcd2000 = preview_table(db, "opprcd2000")


=== secnmd ===
['secid', 'effect_date', 'cusip', 'ticker', 'class', 'issuer', 'issue', 'sic']
    secid effect_date     cusip ticker class                      issuer  \
0  5001.0  1996-01-02  00078110  ABSIE  <NA>          ABS INDUSTRIES INC   
1  5001.0  2007-03-08  00078110   ZZZZ  <NA>                ABS INDS INC   
2  5002.0  1996-01-01  00103010  AELNA  <NA>       AEL INDUSTRIES - CI A   
3  5003.0  1996-01-01  00103810   AFAP  <NA>  AFA PROTECTIVE SYSTEMS INC   
4  5003.0  1999-07-08  00103810  AFAPE  <NA>  AFA PROTECTIVE SYSTEMS INC   

  issue   sic  
0  <NA>  <NA>  
1   COM  3462  
2  <NA>  <NA>  
3  <NA>  <NA>  
4  <NA>  <NA>  

=== optionmnames ===
['secid', 'symbol', 'optionid', 'root', 'suffix', 'effect_date', 'cusip', 'ticker', 'class', 'issuer', 'issue']
    secid symbol optionid  root suffix effect_date     cusip ticker class  \
0  5001.0   <NA>     <NA>  <NA>   <NA>  1996-01-02  00078110  ABSIE  <NA>   
1  5001.0   <NA>     <NA>  <NA>   <NA>  2007-03-08  00078110   Z

In [2]:
def extract_optionm_secnmd(
    db,
    start_date="2000-01-01",
    output_file="data/optionm_secnmd.csv",
):
    df = db.raw_sql(f"""
        select
            secid,
            effect_date,
            cusip,
            ticker,
            class,
            issuer,
            issue,
            sic
        from optionm_all.secnmd
        where effect_date >= '{start_date}'
    """, date_cols=["effect_date"])

    for col in ["cusip", "ticker", "class", "issuer", "issue"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip().str.upper()

    df["cusip8"] = df["cusip"].str[:8]
    df["cusip6"] = df["cusip"].str[:6]

    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_file, index=False)
    return df

secnmd = extract_optionm_secnmd(
    db,
    start_date="2000-01-01",
    output_file="data/optionm_secnmd_2000_2025.csv",
)
print(secnmd.head())
print(secnmd.shape)

    secid effect_date     cusip ticker class                       issuer  \
0  5001.0  2007-03-08  00078110   ZZZZ  <NA>                 ABS INDS INC   
1  5003.0  2009-12-31  00103810   AFAP  <NA>             AFA PROT SYS INC   
2  5003.0  2021-08-09  00103810   AFAP  <NA>  AFA PROTECTIVE SYSTEMS INC.   
3  5007.0  2001-04-09  00495010    ACK  <NA>                 ACKTION CORP   
4  5007.0  2002-08-02  61757710    MRC  <NA>                MORGUARD CORP   

             issue   sic    cusip8  cusip6  
0              COM  3462  00078110  000781  
1              COM  7382  00103810  001038  
2  ORDINARY SHARES  7380  00103810  001038  
3              COM  6512  00495010  004950  
4              COM  6512  61757710  617577  
(248610, 10)


In [3]:
def build_crsp_optionm_link(
    crsp_file="data/crsp_id_master_2000_2025.csv",
    secnmd_file="data/optionm_secnmd.csv",
    output_file="data/crsp_optionm_link.csv",
    include_ticker_fallback=False,
):
    crsp = pd.read_csv(crsp_file, parse_dates=["namedt", "nameenddt"])
    opt = pd.read_csv(secnmd_file, parse_dates=["effect_date"])

    # standardize strings
    for df in (crsp, opt):
        for col in df.columns:
            if df[col].dtype == object or str(df[col].dtype).startswith("string"):
                df[col] = df[col].astype("string").str.strip().str.upper()

    # CRSP historical CUSIP preference
    if "cusip_hist" not in crsp.columns:
        crsp["cusip_hist"] = crsp["ncusip"]
        if "cusip" in crsp.columns:
            crsp["cusip_hist"] = crsp["cusip_hist"].fillna(crsp["cusip"])

    crsp["cusip8"] = crsp["cusip_hist"].astype("string").str[:8]

    # keep only valid non-missing cusips
    crsp_c = crsp[crsp["cusip8"].notna() & (crsp["cusip8"] != "")]
    opt_c = opt[opt["cusip8"].notna() & (opt["cusip8"] != "")]

    # primary link: CUSIP8
    link_cusip = crsp_c.merge(
        opt_c,
        on="cusip8",
        how="inner",
        suffixes=("_crsp", "_opt")
    ).copy()
    link_cusip["link_method"] = "cusip8"

    # choose one record per permno-secid pair, earliest dates first
    link_cusip = (
        link_cusip
        .sort_values(["permno", "secid", "namedt", "effect_date"])
        .drop_duplicates(subset=["permno", "secid"], keep="first")
        .reset_index(drop=True)
    )

    pieces = [link_cusip]

    if include_ticker_fallback:
        crsp_t = crsp[crsp["ticker"].notna() & (crsp["ticker"] != "")]
        opt_t = opt[opt["ticker"].notna() & (opt["ticker"] != "")]

        link_ticker = crsp_t.merge(
            opt_t,
            on="ticker",
            how="inner",
            suffixes=("_crsp", "_opt")
        ).copy()
        link_ticker["link_method"] = "ticker"

        # drop permno-secid already captured by cusip
        used = set(zip(link_cusip["permno"], link_cusip["secid"]))
        link_ticker = link_ticker[
            ~link_ticker.apply(lambda r: (r["permno"], r["secid"]) in used, axis=1)
        ].copy()

        link_ticker = (
            link_ticker
            .sort_values(["permno", "secid", "namedt", "effect_date"])
            .drop_duplicates(subset=["permno", "secid"], keep="first")
            .reset_index(drop=True)
        )

        pieces.append(link_ticker)

    link = pd.concat(pieces, ignore_index=True, sort=False)

    # optional: keep one dominant secid per permno for clean baseline
    # choose CUSIP over ticker, then earliest link dates
    link["link_priority"] = link["link_method"].map({"cusip8": 1, "ticker": 2}).fillna(99)

    dominant = (
        link
        .sort_values(["permno", "link_priority", "namedt", "effect_date"])
        .drop_duplicates(subset=["permno"], keep="first")
        .reset_index(drop=True)
    )

    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    link.to_csv(output_file, index=False)

    dominant_file = Path(output_file).with_name("crsp_optionm_link_dominant.csv")
    dominant.to_csv(dominant_file, index=False)

    return link, dominant

In [4]:
link_all, link_dom = build_crsp_optionm_link(
    crsp_file="data/crsp_id_master_2000_2025.csv",
    secnmd_file="data/optionm_secnmd_2000_2025.csv",
    output_file="data/crsp_optionm_link.csv",
    include_ticker_fallback=False,
)

In [3]:
from pathlib import Path
import pandas as pd
import duckdb
from tqdm.auto import tqdm


def fetch_opprcd_yearly_for_linked_secid(
    db,
    link_file="data/crsp_optionm_link_dominant.csv",
    data_dir="data",
    start_year=2000,
    end_year=2025,
    secid_chunk_size=200,
    replace=False,
    final_output_file="opprcd_linked_2000_2025.parquet",
    compression="zstd",
    keep_chunk_csv=True,
):
    link = pd.read_csv(link_file)
    secid_list = sorted(link["secid"].dropna().astype(int).unique().tolist())

    chunk_dir = Path(data_dir) / "opprcd_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)

    final_output_path = Path(data_dir) / final_output_file
    final_output_path.parent.mkdir(parents=True, exist_ok=True)

    select_cols = [
        "secid",
        "date",
        "exdate",
        "cp_flag",
        "strike_price",
        "best_bid",
        "best_offer",
        "volume",
        "open_interest",
        "impl_volatility",
        "delta",
        "gamma",
        "vega",
        "theta",
        "optionid",
        "cfadj",
        "contract_size",
        "ss_flag",
        "forward_price",
        "expiry_indicator",
        "root",
        "suffix",
    ]

    chunk_specs = []
    for year in range(start_year, end_year + 1):
        for chunk_idx, start_i in enumerate(range(0, len(secid_list), secid_chunk_size)):
            end_i = min(start_i + secid_chunk_size, len(secid_list))
            chunk_specs.append((year, chunk_idx, start_i, end_i))

    outer_pbar = tqdm(chunk_specs, desc="opprcd chunks", unit="chunk")

    for year, chunk_idx, start_i, end_i in outer_pbar:
        sub = secid_list[start_i:end_i]
        chunk_file = chunk_dir / f"opprcd_{year}_chunk_{chunk_idx:05d}.csv"

        if chunk_file.exists() and not replace:
            outer_pbar.set_postfix_str(f"skip {year} c{chunk_idx:05d}")
            continue

        table_name = f"opprcd{year}"
        in_clause = ",".join(str(int(x)) for x in sub)

        sql = f"""
            select {", ".join(select_cols)}
            from optionm_all.{table_name}
            where secid in ({in_clause})
        """

        df_part = db.raw_sql(sql, date_cols=["date", "exdate"])
        df_part.to_csv(chunk_file, index=False)
        outer_pbar.set_postfix_str(f"saved {year} c{chunk_idx:05d}")

    if final_output_path.exists() and not replace:
        return final_output_path

    chunk_files = sorted(chunk_dir.glob("opprcd_*.csv"))
    if not chunk_files:
        raise FileNotFoundError(f"No chunk CSV files found in {chunk_dir}")

    csv_glob = chunk_dir / "opprcd_*.csv"

    con = duckdb.connect()
    try:
        con.execute(f"""
            COPY (
                SELECT
                    CAST(secid AS BIGINT) AS secid,
                    CAST(date AS DATE) AS date,
                    CAST(exdate AS DATE) AS exdate,
                    cp_flag,
                    CAST(strike_price AS DOUBLE) AS strike_price,
                    CAST(best_bid AS DOUBLE) AS best_bid,
                    CAST(best_offer AS DOUBLE) AS best_offer,
                    CAST(volume AS DOUBLE) AS volume,
                    CAST(open_interest AS DOUBLE) AS open_interest,
                    CAST(impl_volatility AS DOUBLE) AS impl_volatility,
                    CAST(delta AS DOUBLE) AS delta,
                    CAST(gamma AS DOUBLE) AS gamma,
                    CAST(vega AS DOUBLE) AS vega,
                    CAST(theta AS DOUBLE) AS theta,
                    CAST(optionid AS BIGINT) AS optionid,
                    CAST(cfadj AS DOUBLE) AS cfadj,
                    CAST(contract_size AS DOUBLE) AS contract_size,
                    CAST(ss_flag AS INTEGER) AS ss_flag,
                    CAST(forward_price AS DOUBLE) AS forward_price,
                    expiry_indicator,
                    root,
                    suffix
                FROM read_csv_auto(
                    '{csv_glob.as_posix()}',
                    union_by_name = true,
                    header = true
                )
                ORDER BY date, secid, exdate, optionid
            )
            TO '{final_output_path.as_posix()}'
            (FORMAT PARQUET, COMPRESSION '{compression}')
        """)
    finally:
        con.close()

    if not keep_chunk_csv:
        for f in chunk_files:
            f.unlink(missing_ok=True)

    return final_output_path

In [ ]:
parquet_path = fetch_opprcd_yearly_for_linked_secid(
    db,
    link_file="data/crsp_optionm_link_dominant.csv",
    data_dir="data",
    start_year=2024,
    end_year=2024,
    secid_chunk_size=25,
    replace=False,
    final_output_file="opprcd_linked.parquet",
    compression="zstd",
    keep_chunk_csv=True,
)
print(parquet_path)

opprcd chunks:   0%|          | 0/491 [00:00<?, ?chunk/s]